# L4: Ferramentas para uma campanha de divulgação do cliente
Nesta lição, você aprenderá mais sobre Ferramentas. Você se concentrará em três elementos principais das Ferramentas:

* Versatilidade
* Tolerância a falhas
* Cache

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew

In [3]:
import os
from utils import get_openai_api_key, pretty_print_result
from utils import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

# Creating Agents

In [4]:
sales_rep_agent = Agent(
    role="Representante de Vendas",
    goal="Identificar leads de alto valor que correspondam "
         "ao nosso perfil ideal de cliente",
    backstory=(
        "Como parte da equipe dinâmica de vendas da CrewAI, "
        "sua missão é explorar o "
        "cenário digital em busca de potenciais leads. "
        "Munido de ferramentas de ponta "
        "e uma mentalidade estratégica, você analisa dados, "
        "tendências e interações para "
        "descobrir oportunidades que outros podem não perceber. "
        "Seu trabalho é crucial para abrir caminho "
        "para interações significativas e impulsionar o crescimento da empresa."
    ),
    allow_delegation=False,
    verbose=True
)

LLM value is None


In [5]:
lead_sales_rep_agent = Agent(
    role="Representante Líder de Vendas",
    goal="Cultivar leads com comunicações personalizadas e atraentes",
    backstory=(
        "Dentro do vibrante ecossistema do departamento de vendas da CrewAI, "
        "você se destaca como a ponte entre clientes potenciais "
        "e as soluções de que eles precisam."
        "Ao criar mensagens envolventes e personalizadas, "
        "você não apenas informa os leads sobre nossas ofertas, "
        "mas também os faz se sentirem vistos e ouvidos."
        "Seu papel é fundamental para converter interesse "
        "em ação, guiando os leads na jornada "
        "da curiosidade ao compromisso."
    ),
    allow_delegation=False,
    verbose=True
)

LLM value is None


# Creating Tools

crewAI Tools

In [6]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [7]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

Custom Tool
* Crie uma ferramenta personalizada usando a classe BaseTool do CrewAi

In [8]:
from crewai.tools import BaseTool

* Cada ferramenta precisa ter um nome e uma descrição.
* Para simplificar e para fins de sala de aula, SentimentAnalysisTool retornará positivo para cada texto.
* Ao executar localmente, você pode personalizar o código com sua lógica na função _run.

In [9]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [10]:
sentiment_analysis_tool = SentimentAnalysisTool()

# Creating Tasks
* A tarefa de criação de perfil de lead usa ferramentas CrewAI.

In [11]:
lead_profiling_task = Task(
    description=(
        "Conduza uma análise aprofundada de {lead_name}, "
        "uma empresa do setor {industry} "
        "que recentemente demonstrou interesse em nossas soluções. "
        "Utilize todas as fontes de dados disponíveis "
        "para compilar um perfil detalhado, "
        "focando nos principais tomadores de decisão, "
        "desenvolvimentos recentes nos negócios e necessidades potenciais "
        "que se alinhem com nossas ofertas. "
        "Esta tarefa é crucial para adaptar "
        "nossa estratégia de engajamento de forma eficaz.\n"
        "Não faça suposições e "
        "use apenas informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Um relatório abrangente sobre {lead_name}, "
        "incluindo histórico da empresa, "
        "pessoal-chave, marcos recentes e necessidades identificadas. "
        "Destaque áreas potenciais onde "
        "nossas soluções possam agregar valor "
        "e sugira estratégias de engajamento personalizadas."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)


* A tarefa de divulgação personalizada está usando sua ferramenta personalizada SentimentAnalysisTool, bem como SerperDevTool da tripulaçãoAI (search_tool).

In [12]:
personalized_outreach_task = Task(
    description=(
        "Usando os insights obtidos a partir "
        "do relatório de perfil de leads sobre {lead_name}, "
        "crie uma campanha de contato personalizada "
        "destinada a {key_decision_maker}, "
        "{position} de {lead_name}. "
        "A campanha deve abordar o recente {milestone} deles "
        "e como nossas soluções podem apoiar seus objetivos. "
        "Sua comunicação deve ressoar "
        "com a cultura e os valores da empresa {lead_name}, "
        "demonstrando um profundo entendimento do "
        "negócio e das necessidades deles.\n"
        "Não faça suposições e use apenas "
        "informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Uma série de rascunhos de e-mails personalizados "
        "adaptados para {lead_name}, "
        "especificamente direcionados a {key_decision_maker}. "
        "Cada rascunho deve incluir "
        "uma narrativa convincente que conecte nossas soluções "
        "aos recentes resultados e objetivos futuros deles. "
        "Garanta que o tom seja envolvente, profissional "
        "e alinhado com a identidade corporativa de {lead_name}."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

# Creating the Crew

In [13]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

Running the Crew

In [14]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Plataforma de Aprendizado Online",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "lançamento de produto"
}

result = crew.kickoff(inputs=inputs)

# Agent: Representante de Vendas
## Task: Conduza uma análise aprofundada de DeepLearningAI, uma empresa do setor Plataforma de Aprendizado Online que recentemente demonstrou interesse em nossas soluções. Utilize todas as fontes de dados disponíveis para compilar um perfil detalhado, focando nos principais tomadores de decisão, desenvolvimentos recentes nos negócios e necessidades potenciais que se alinhem com nossas ofertas. Esta tarefa é crucial para adaptar nossa estratégia de engajamento de forma eficaz.
Não faça suposições e use apenas informações sobre as quais tenha absoluta certeza.


# Agent: Representante de Vendas
## Using tool: List files in directory
## Tool Input: 
"{}"
## Tool Output: 
File paths: 
-


# Agent: Representante de Vendas
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./instructions/DeepLearningAI_Historical_Data.txt\"}"
## Tool Output: 
Error: File not found at path: ./instructions/DeepLearningAI_Historical_Data.txt


# Agent: Repre

In [17]:
from IPython.display import Markdown

texto_markdown = result.tasks_output[0].raw  
Markdown(texto_markdown)

DeepLearningAI é uma empresa fundada em 2017 por Andrew Ng, pioneiro em aprendizado de máquina e educação, com o objetivo de fornecer educação em IA de classe mundial. A empresa criou programas de IA de alta qualidade na plataforma Coursera, ganhando uma ampla aceitação global.

Principais Pessoas-Chave:
- Andrew Ng: Fundador da DeepLearningAI, Managing General Partner do AI Fund e Executive Chairman da Landing AI. Reconhecido como uma influência no campo da IA.

Desenvolvimentos Recentes nos Negócios:
- A DeepLearningAI continua a fornecer cursos e especializações em IA para mais de 7 milhões de pessoas, visando capacitar indivíduos a utilizar e construir soluções em IA.
- A empresa está focada em oferecer treinamento prático e colaborativo para criar uma comunidade de aprendizagem.

Necessidades Identificadas:
- A DeepLearningAI busca atender à demanda global por educação em IA de alta qualidade, visando capacitar profissionais e entusiastas de tecnologia em novas habilidades e conhecimentos.
- Há um foco em expandir as oportunidades de aplicação da IA em diversas áreas, buscando ampliar a adoção e compreensão da tecnologia.

Áreas Potenciais para Nossas Soluções Agregarem Valor:
- Nossas soluções de aprendizado automatizado e análise de dados podem auxiliar a DeepLearningAI a aprimorar seus métodos de ensino, personalizando o aprendizado e otimizando os resultados dos alunos.
- A nossa plataforma de interação dinâmica pode ser integrada aos cursos da DeepLearningAI para promover um engajamento mais significativo e colaborativo entre os alunos.

Estratégias de Engajamento Personalizadas Sugeridas:
- Propor parcerias para desenvolver cursos personalizados que atendam às necessidades específicas da DeepLearningAI e de sua comunidade de alunos.
- Oferecer treinamento especializado para os instrutores da DeepLearningAI, capacitando-os a utilizar tecnologias inovadoras em seus métodos de ensino.

Overall, a DeepLearningAI apresenta um forte comprometimento com a excelência em educação em IA, o que sugere uma forte sinergia com as soluções e abordagens da nossa empresa.